# 06 — Power Spectra Comparison

**Purpose:** Compare the two-point statistics of Agora test maps, DDPM-generated samples,
and Gaussian realisations.

This notebook computes and plots the angular auto- and cross-power spectra for the CIB
and tSZ channels:

1. **Auto-spectra** — Cℓ^{CIB} and Cℓ^{tSZ} for all three sample types, binned over
   300 < ℓ < 4000 in bins of Δℓ = 60.

2. **Cross-spectrum** — Cℓ^{CIB×tSZ} probing spatial correlation between dusty
   star-forming galaxies and hot gas traced by tSZ.

3. **Residuals** — (C_ℓ^{Agora} − C_ℓ^{DDPM}) / σ(C_ℓ^{Agora}) to quantify agreement
   within sample variance.

4. **Multi-frequency correlations** — cross-correlation coefficients between the 95, 150,
   and 857 GHz CIB channels (Appendix B).

**Inputs:**
- Test maps and DDPM samples: `data/low_pass/2mJy/*.npy`
- Gaussian baseline: `data/low_pass/2mJy/gaussian_cib_tsz_2mJy_lp.npy`
- ILC noise power spectra: `data/ilc/ilc_weights_residuals_agora_fg_model.npy`

**Outputs:** power spectrum comparison plots (Figure 4), multi-frequency correlation
plots (Figure 9).

**Key module functions:**
- `foregrounds_diffusion.flatmaps.map2cl`
- `foregrounds_diffusion.flatmaps.cl2map`
- `foregrounds_diffusion.preprocessing.renormalize_dm_maps`

**Paper reference:** §4.3 (Figure 4), Appendix B (Figure 9).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from foregrounds_diffusion.flatmaps import map2cl
from foregrounds_diffusion.preprocessing import renormalize_dm_maps

PTSRC = 2
flatskymapparams = [256, 256, 1.41, 1.41]   # [nx, ny, dx_arcmin, dy_arcmin]
LMIN, LMAX, LBIN = 300, 4000, 60


In [ ]:
cib_maps    = np.load(f"data/low_pass/{PTSRC}mJy/CIB_map_150GHz_256_st6_minmax_{PTSRC}mJy_zero_lp.npy")
tsz_maps    = np.load(f"data/low_pass/{PTSRC}mJy/tSZ3_map_150GHz_256_st6_minmax_{PTSRC}mJy_norm_lp.npy")
ddpm_raw    = np.load(f"data/low_pass/{PTSRC}mJy/new_samples_cib_tsz_{PTSRC}mJy_lp.npy")
gauss_maps  = np.load(f"data/low_pass/{PTSRC}mJy/gaussian_cib_tsz_{PTSRC}mJy_lp.npy")

# DDPM samples are (N, 2, H, W) channels-first; Agora/Gaussian are (N, H, W, 1)
ddpm_cib = ddpm_raw[:, 0]   # (N, H, W)
ddpm_tsz = ddpm_raw[:, 1]

agora_cib  = cib_maps[:, :, :, 0]
agora_tsz  = tsz_maps[:, :, :, 0]

# gauss_maps shape depends on generation; handle both (N, 2, H, W) and (N, H, W, 2)
if gauss_maps.ndim == 4 and gauss_maps.shape[1] == 2:
    gauss_cib = gauss_maps[:, 0]
    gauss_tsz = gauss_maps[:, 1]
else:
    gauss_cib = gauss_maps[:, :, :, 0]
    gauss_tsz = gauss_maps[:, :, :, 1]

N = min(len(agora_cib), len(ddpm_cib), len(gauss_cib))
print(f"Using {N} maps from each source for power spectrum estimation")


In [ ]:
def mean_cls(maps_nhw, mapparams, lmin, lmax, binsize):
    """Compute mean auto-power spectrum over N maps, returning (el, mean_cl, std_cl)."""
    cls = []
    for m in maps_nhw:
        el, cl = map2cl(mapparams, m, binsize=binsize, minbin=lmin, maxbin=lmax)
        cls.append(cl)
    cls = np.array(cls)
    return el, cls.mean(axis=0), cls.std(axis=0)

def mean_cross_cls(maps1, maps2, mapparams, lmin, lmax, binsize):
    cls = []
    for m1, m2 in zip(maps1, maps2):
        el, cl = map2cl(mapparams, m1, m2, binsize=binsize, minbin=lmin, maxbin=lmax)
        cls.append(cl)
    cls = np.array(cls)
    return el, cls.mean(axis=0), cls.std(axis=0)

el, cl_cib_agora,  cl_cib_agora_err  = mean_cls(agora_cib[:N],  flatskymapparams, LMIN, LMAX, LBIN)
_,  cl_cib_ddpm,   cl_cib_ddpm_err   = mean_cls(ddpm_cib[:N],   flatskymapparams, LMIN, LMAX, LBIN)
_,  cl_cib_gauss,  cl_cib_gauss_err  = mean_cls(gauss_cib[:N],  flatskymapparams, LMIN, LMAX, LBIN)

_,  cl_tsz_agora,  cl_tsz_agora_err  = mean_cls(agora_tsz[:N],  flatskymapparams, LMIN, LMAX, LBIN)
_,  cl_tsz_ddpm,   cl_tsz_ddpm_err   = mean_cls(ddpm_tsz[:N],   flatskymapparams, LMIN, LMAX, LBIN)
_,  cl_tsz_gauss,  cl_tsz_gauss_err  = mean_cls(gauss_tsz[:N],  flatskymapparams, LMIN, LMAX, LBIN)

_,  cl_cross_agora,  _  = mean_cross_cls(agora_cib[:N], agora_tsz[:N], flatskymapparams, LMIN, LMAX, LBIN)
_,  cl_cross_ddpm,   _  = mean_cross_cls(ddpm_cib[:N],  ddpm_tsz[:N],  flatskymapparams, LMIN, LMAX, LBIN)
_,  cl_cross_gauss,  _  = mean_cross_cls(gauss_cib[:N], gauss_tsz[:N], flatskymapparams, LMIN, LMAX, LBIN)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, cl_a, err_a, cl_d, err_d, cl_g, title in zip(
    axes,
    [cl_cib_agora,   cl_tsz_agora,   cl_cross_agora],
    [cl_cib_agora_err, cl_tsz_agora_err, np.zeros_like(cl_cross_agora)],
    [cl_cib_ddpm,    cl_tsz_ddpm,    cl_cross_ddpm],
    [cl_cib_ddpm_err,  cl_tsz_ddpm_err,  np.zeros_like(cl_cross_ddpm)],
    [cl_cib_gauss,   cl_tsz_gauss,   cl_cross_gauss],
    [r"$C_\ell^{\rm CIB}$", r"$C_\ell^{\rm tSZ}$", r"$C_\ell^{\rm CIB \times tSZ}$"],
):
    ax.fill_between(el, cl_a - err_a, cl_a + err_a, alpha=0.3, color="C0")
    ax.plot(el, cl_a, label="Agora", color="C0")
    ax.plot(el, cl_d, label="DDPM",  color="C1")
    ax.plot(el, cl_g, label="Gaussian", color="C2", linestyle="--")
    ax.set_xlabel(r"$\ell$");  ax.set_ylabel(title);  ax.legend()
    ax.set_yscale("log")

plt.tight_layout()
plt.show()


In [ ]:
# Fractional residuals normalised by Agora sample variance
resid_cib = (cl_cib_agora - cl_cib_ddpm) / (cl_cib_agora_err + 1e-30)
resid_tsz = (cl_tsz_agora - cl_tsz_ddpm) / (cl_tsz_agora_err + 1e-30)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.axhline(0, color="k", lw=0.8, ls="--")
ax1.plot(el, resid_cib)
ax1.set_xlabel(r"$\ell$");  ax1.set_ylabel(r"$(C_\ell^{\rm Agora} - C_\ell^{\rm DDPM}) / \sigma$")
ax1.set_title("CIB residuals")

ax2.axhline(0, color="k", lw=0.8, ls="--")
ax2.plot(el, resid_tsz)
ax2.set_xlabel(r"$\ell$");  ax2.set_title("tSZ residuals")
plt.tight_layout();  plt.show()
